## Imported

In [ ]:
import numpy as np
import csv
import random
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    AutoConfig
)

## Generate dataset

In [ ]:
import random
import pandas as pd

random.seed(42)

prefixes = ["", "tolong ", "coba ", "robot ", "hei ", "bot ", "minta tolong "]
suffixes = ["", " ya", " dong", " sekarang", " sebentar", " di sana"]

# NAVIGATE_TO_OBJECT
objects = [
    "meja", "kursi", "pintu", "laptop", "botol", "lemari", "kulkas",
    "televisi", "papan tulis", "kasur", "sofa", "dispenser", "tempat sampah",
    "gelas", "sepatu", "tas", "buku", "proyektor", "jendela", "kamarmandi"
]
obj_positions = ["", " di depan", " di sana", " sebelah kiri", " sebelah kanan", " dekat dinding"]

# MOVE_RELATIVE
rel_verbs = ["jalan", "maju", "mundur", "geser", "bergerak", "pindah"]
rel_directions = ["ke depan", "ke belakang", "ke kiri", "ke kanan", "lurus"]
distances = ["1", "2", "3", "0.5", "setengah", "sedikit", "beberapa"]
units = ["meter", "m", "cm", "langkah"]

# untuk ROTATE
rot_verbs = ["putar", "berputar", "rotasi", "belok", "mengarahkan badan", "tengok", "hadap"]
rot_directions = ["ke kiri", "ke kanan", "ke belakang", "kiri", "kanan", "belakang"]
degrees = ["30", "45", "60", "90", "180", "360"]

def generate_navigate_samples(count=100):
    samples = set()
    actions = ["jalan ke", "pergi ke", "menuju ke", "samperin", "dekati", "bergerak ke", "cari"]

    while len(samples) < count:
        p = random.choice(prefixes)
        a = random.choice(actions)
        o = random.choice(objects)
        pos = random.choice(obj_positions)
        s = random.choice(suffixes)

        sentence = f"{p}{a} {o}{pos}{s}".strip().lower()

        sentence = " ".join(sentence.split())
        samples.add(sentence)

    return list(samples)

def generate_move_relative_samples(count=100):
    samples = set()

    while len(samples) < count:
        p = random.choice(prefixes)
        v = random.choice(rel_verbs)
        d = random.choice(rel_directions)
        dist = random.choice(distances)
        u = random.choice(units)
        s = random.choice(suffixes)

        pattern = random.choice([1, 2, 3])
        if pattern == 1:
            sentence = f"{p}{v} {d} {dist} {u}{s}"
        elif pattern == 2:
            sentence = f"{p}{v} {d}{s}"
        else:
            sentence = f"{p}{v} {dist} {u} {d}{s}"

        sentence = " ".join(sentence.strip().lower().split())
        samples.add(sentence)

    return list(samples)

def generate_rotate_samples(count=100):
    samples = set()

    while len(samples) < count:
        p = random.choice(prefixes)
        v = random.choice(rot_verbs)
        d = random.choice(rot_directions)
        deg = random.choice(degrees)
        s = random.choice(suffixes)

        pattern = random.choice([1, 2, 3])
        if pattern == 1:
            sentence = f"{p}{v} {d} {deg} derajat{s}"
        elif pattern == 2:
            sentence = f"{p}{v} {d}{s}"
        else:
            sentence = f"{p}{v} {deg} derajat {d}{s}"

        sentence = " ".join(sentence.strip().lower().split())
        samples.add(sentence)

    return list(samples)

print("Menghasilkan dataset...")
nav_data = generate_navigate_samples(100)
move_data = generate_move_relative_samples(100)
rot_data = generate_rotate_samples(100)

all_texts = nav_data + move_data + rot_data
all_labels = [0] * 100 + [1] * 100 + [2] * 100

data_pairs = list(zip(all_texts, all_labels))
random.shuffle(data_pairs)

df = pd.DataFrame(data_pairs, columns=["text", "label"])

csv_filename = "dataset_robot_slam.csv"
df.to_csv(csv_filename, index=False)

print(f"Selesai! {len(df)} baris dataset berhasil disimpan di '{csv_filename}'.")
print("\nPratinjau 5 data pertama:")
print(df.head())

Menghasilkan dataset...
Selesai! 300 baris dataset berhasil disimpan di 'dataset_robot_slam.csv'.

Pratinjau 5 data pertama:
                                     text  label
0                      coba putar ke kiri      2
1            bot samperin sofa di sana ya      0
2                   maju ke kanan di sana      1
3  tolong pindah lurus 0.5 meter sebentar      1
4     tolong samperin tas sebelah kiri ya      0


## Preprocess data

In [ ]:
data = np.genfromtxt('dataset_robot_slam.csv', delimiter=',', dtype=None, names=True, encoding='utf-8')

data = {
    'text': data['text'],
    'label': data['label']
}

id2label = {0: "NAVIGATE_TO_OBJECT", 1: "MOVE_RELATIVE", 2: "ROTATE"}
label2id = {"NAVIGATE_TO_OBJECT": 0, "MOVE_RELATIVE": 1, "ROTATE": 2}

df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(test_size=0.2, seed=42)
MODEL_NAME = "indolem/indobert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=32)

tokenized_datasets = dataset.map(preprocess_function, batched=True)

config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = 3
config.id2label = id2label
config.label2id = label2id

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True
)

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  445MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Training Process

In [ ]:
training_args = TrainingArguments(
    output_dir="./indobert_robot_intent",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
)

print("Memulai Fine-Tuning IndoBERT...")
trainer.train()
print("Selesai Fine-Tuning IndoBERT!")

output_model_dir = "./saved_indobert_intent_model"
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print(f"Model berhasil disimpan di folder: {output_model_dir}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Memulai Fine-Tuning IndoBERT...


Epoch,Training Loss,Validation Loss
1,0.936954,0.799496
2,0.678021,0.614802
3,0.454879,0.312000
4,0.168577,0.049958
5,0.161997,0.045563
6,0.034093,0.035124
7,0.006659,0.003636
8,0.013244,0.001438
9,0.002599,0.001365
10,0.002114,0.001314


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai Fine-Tuning IndoBERT!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model berhasil disimpan di folder: ./saved_indobert_intent_model


## Test

In [ ]:
from transformers import pipeline

nlp_classifier = pipeline(
    "text-classification",
    model="./saved_indobert_intent_model",
    tokenizer="./saved_indobert_intent_model"
)

kalimat_uji = "Dekati buku didepan mu"
hasil = nlp_classifier(kalimat_uji)

print(f"Kalimat : '{kalimat_uji}'")
print(f"Intent  : {hasil[0]['label']}")
print(f"Score   : {hasil[0]['score']:.4f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Kalimat : 'Dekati buku didepan mu'
Intent  : NAVIGATE_TO_OBJECT
Score   : 0.9986
